# Sesame CSM: sentence to voice in Google Colab

Type a sentence, generate speech, listen, and download a WAV file. The model stays loaded between requests.

**Credits:** This notebook is part of [darkluminance/csm-windows](https://github.com/darkluminance/csm-windows), derived from [SesameAILabs/csm](https://github.com/SesameAILabs/csm). The original model and code are by Sesame AI Labs and the upstream authors, under the [Apache 2.0 license](https://github.com/SesameAILabs/csm/blob/main/LICENSE). Audio watermarking is preserved.

## Before you begin

1. Choose **Runtime → Change runtime type → GPU**, then connect. GPU availability depends on your Colab plan and current availability. T4 is supported using FP16; GPUs supporting BF16 use BF16. CPU runtimes are not the intended notebook workflow.
2. Gain access to both [Sesame CSM-1B](https://huggingface.co/sesame/csm-1b) and [Llama-3.2-1B](https://huggingface.co/meta-llama/Llama-3.2-1B). Wait until **both** requests are approved.
3. Create a [Hugging Face read token](https://huggingface.co/settings/tokens) on the approved account. For a fine-grained token, permit access to both gated repositories.
4. Run cells **1–4 individually, in order**, then use the Generate button. Do not use **Run all**: the download, log, and stop cells are optional manual actions. The first setup/download can take several minutes and uses several GB of disk. Save generated audio before disconnecting: Colab runtime files are temporary.

Optional: open Colab's **Secrets** panel (key icon), add a secret named `HF_TOKEN`, and enable notebook access. Otherwise cell 2 prompts privately. **Never paste your token into a code cell or share notebook outputs containing credentials.**

This notebook uses an isolated Python 3.11 environment for the pinned model stack. The Colab notebook kernel only handles widgets and local HTTP requests; it does not import CSM. The UI runs inside the notebook; no public web tunnel is created.


## 1. Install the model environment

Run once per fresh runtime. This clones the Windows edition's source (which also supports Linux), installs Linux CUDA packages and FFmpeg, and checks dependency consistency. It does **not** install `triton-windows`. Installing into a separate environment avoids replacing Colab's preinstalled PyTorch packages. You normally do not need to restart the notebook kernel.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

if not Path('/content').is_dir():
    raise RuntimeError('Open this notebook in Google Colab before running setup.')
try:
    subprocess.run(['nvidia-smi'], check=True)
except (FileNotFoundError, subprocess.CalledProcessError) as error:
    raise RuntimeError('Select a GPU under Runtime → Change runtime type, reconnect, and rerun this cell.') from error

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'uv', 'ipywidgets>=8,<9', 'requests'])
repo = Path('/content/csm-windows')
if not repo.exists():
    subprocess.check_call(['git', 'clone', '--depth', '1', 'https://github.com/darkluminance/csm-windows.git', str(repo)])
else:
    print('Reusing the checkout from this runtime.')
python = repo / '.venv/bin/python'
uv = [sys.executable, '-m', 'uv']
if not python.exists():
    subprocess.check_call(uv + ['venv', '--python', '3.11', str(repo / '.venv')])
subprocess.check_call(uv + ['pip', 'install', '--python', str(python), 'torch==2.4.0', 'torchaudio==2.4.0', '--index-url', 'https://download.pytorch.org/whl/cu124'])
subprocess.check_call(uv + ['pip', 'install', '--python', str(python), '-r', str(repo / 'requirements-colab.txt')], cwd=repo)
subprocess.check_call(uv + ['pip', 'check', '--python', str(python)])
subprocess.check_call(['apt-get', '-qq', 'update'])
subprocess.check_call(['apt-get', '-qq', 'install', '-y', 'ffmpeg'])
subprocess.check_call([str(python), '-c', "import torch; assert torch.cuda.is_available(), 'CUDA is unavailable'; print('GPU:', torch.cuda.get_device_name(0)); print('Native BF16:', torch.cuda.is_bf16_supported(including_emulation=False))"])
print('Setup complete. Continue to authentication.')


## 2. Authenticate and check both model approvals

Use the token belonging to the account with approved access. This cell first tries the `HF_TOKEN` Colab secret, then falls back to a hidden input prompt. The token is passed privately to the model process and is not printed or written to a repository file. Login does not replace model access approval.


In [ ]:
from getpass import getpass

token = None
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
except Exception:
    pass  # Missing/disabled secret: use a hidden prompt instead.
if not token:
    token = getpass('Hugging Face read token (hidden): ').strip()
if not token:
    raise ValueError('A Hugging Face read token is required.')

model_env = dict(os.environ)
model_env['HF_TOKEN'] = token
model_env['NO_TORCH_COMPILE'] = '1'
del token
subprocess.check_call([str(python), str(repo / 'check_model_access.py')], env=model_env, cwd=repo)
print('Both model repositories are accessible. Continue to loading.')


## 3. Load the model before using the interface

Run this cell and wait for **Model ready**. The first run downloads CSM, the Llama tokenizer, Mimi, and watermarking weights. Later sentences reuse this model without loading it again. The backend listens only inside your runtime on localhost. Rerunning this cell stops its previous backend and reloads the model.

If this fails, inspect the log cell under troubleshooting. Do not run the UI until this cell succeeds.


In [ ]:
import time
import requests

if 'server' in globals() and server.poll() is None:
    server.terminate()
    server.wait(timeout=30)
if 'server_log' in globals():
    server_log.close()
log_path = Path('/content/csm-server.log')
server_log = log_path.open('w')
server = subprocess.Popen([str(python), '-u', str(repo / 'app.py')], cwd=repo, env=model_env, stdout=server_log, stderr=subprocess.STDOUT)
base_url = 'http://127.0.0.1:7860'
for attempt in range(120):
    if server.poll() is not None:
        raise RuntimeError('The backend exited. Run the troubleshooting log cell below.')
    try:
        if requests.get(base_url, timeout=2).ok:
            break
    except requests.RequestException:
        pass
    time.sleep(1)
else:
    raise RuntimeError('The backend did not start. Inspect the log below.')

print('Loading model and downloading any missing weights. Please wait…', flush=True)
response = requests.post(base_url + '/prepare', timeout=1800)
result = response.json()
if not response.ok:
    raise RuntimeError(result.get('error', 'Model loading failed; inspect the backend log.'))
if result.get('device', '').split(':')[0] != 'cuda':
    raise RuntimeError('The backend selected CPU. Switch to a GPU runtime and rerun setup.')
print('Model ready on GPU. Run the next cell to open the sentence interface.')


## 4. Type a sentence and generate your voice

Run this cell once to display the controls. Type up to **200 characters**, then click **Generate voice**. Start with a short English sentence. The result has an audio player and a **Download WAV** button.

This follows Sesame's basic `speaker=0, context=[]` example: voice identity varies between requests, and no specific person or fixed voice is selected. Output is capped at 30 seconds; shorten your text if it is cut off. The original watermark remains enabled.


In [ ]:
import ipywidgets as widgets
from IPython.display import Audio, display, clear_output
from google.colab import files, output

output.enable_custom_widget_manager()
if 'sentence_ui' in globals():
    sentence_ui.close()
sentence = widgets.Textarea(value='Hello from Sesame.', placeholder='Type a sentence…', description='Sentence:', layout=widgets.Layout(width='100%', height='100px'))
generate_button = widgets.Button(description='Generate voice', button_style='primary')
download_button = widgets.Button(description='Download WAV', disabled=True)
status = widgets.Label(value='Model ready. Enter a sentence.')
audio_output = widgets.Output()
audio_path = Path('/content/speech.wav')

def generate_voice(_):
    text = sentence.value.strip()
    if not text or len(text) > 200:
        status.value = 'Enter a sentence of 1–200 characters.'
        return
    generate_button.disabled = True
    download_button.disabled = True
    status.value = 'Generating your voice…'
    try:
        response = requests.post(base_url + '/generate', json={'text': text}, timeout=600)
        if not response.ok:
            error = response.json().get('error', 'Generation failed. Inspect the backend log.')
            raise RuntimeError(error)
        audio_path.write_bytes(response.content)
        with audio_output:
            clear_output(wait=True)
            display(Audio(filename=str(audio_path)))
        download_button.disabled = False
        status.value = 'Your voice is ready. Play it or download the WAV.'
    except Exception as error:
        status.value = str(error)
    finally:
        generate_button.disabled = False

def download_voice(_):
    if audio_path.exists():
        files.download(str(audio_path))

generate_button.on_click(generate_voice)
download_button.on_click(download_voice)
sentence_ui = widgets.VBox([sentence, widgets.HBox([generate_button, download_button]), status, audio_output])
display(sentence_ui)


## 5. Download again (optional)

If your browser did not start the download from the widget, run this cell. Each generation overwrites `/content/speech.wav`, so download a result before generating another sentence if you want to keep it. You can also find the WAV in Colab's Files panel.


In [ ]:
from google.colab import files
if not Path('/content/speech.wav').exists():
    raise RuntimeError('Generate a sentence first.')
files.download('/content/speech.wav')


## Troubleshooting

- **401/403 / gated repository:** approve both model repositories using the token owner's account. Check read permissions, update your secret/token, then rerun cells 2 and 3.
- **No CUDA / no GPU:** select a GPU under Runtime → Change runtime type and reconnect. GPU availability is not guaranteed. This notebook's setup intentionally stops on a CPU runtime.
- **Out of GPU memory:** stop other model processes in this runtime. Rerun cell 3 to reset this notebook's backend, then try a shorter sentence. If needed, start a fresh GPU runtime.
- **Model load times out:** the initial download can be slow. Check the log below. If the backend is still downloading, wait and rerun the readiness check: `requests.post(base_url + '/prepare', timeout=1800).json()`.
- **Dependency/import error:** use a fresh runtime and rerun setup in order. The notebook uses its own Python environment, so importing `generator` directly in the Colab kernel is not part of this workflow.
- **Blank/unresponsive widgets:** rerun cell 4 and allow Colab's custom widget manager. The model can still be tested with `requests.post(base_url + '/generate', json={'text': 'Hello from Sesame.'}, timeout=600)` from a code cell.
- **Session disconnected:** runtime files/processes may be gone. Reconnect and rerun cells 1–4. Download WAV files before ending a session.
- **Warnings about torch.load or flash attention:** upstream dependencies may emit warnings. Check whether the request succeeds; warnings alone do not imply failure.

The log below contains model/server diagnostics, not your token. Review outputs before sharing a notebook. Generated speech and the text you enter are processed in your own Colab runtime.

References: [Colab FAQ](https://research.google.com/colaboratory/faq.html), [Hugging Face authentication](https://huggingface.co/docs/huggingface_hub/v0.28.1/en/package_reference/authentication), [Jupyter widgets](https://ipywidgets.readthedocs.io/en/stable/examples/Widget%20Basics.html), and [the original CSM README](https://github.com/SesameAILabs/csm).


In [ ]:
log_path = Path('/content/csm-server.log')
print(log_path.read_text()[-12000:] if log_path.exists() else 'No backend log yet. Run cell 3 first.')


## Stop the backend and release GPU memory

Run when finished. This stops only the model process launched by this notebook. Your downloaded audio is unaffected. To use the UI again, rerun cells 3 and 4. To end the Colab session entirely, choose **Runtime → Disconnect and delete runtime** after downloading any files you want to keep.


In [ ]:
if 'server' in globals() and server.poll() is None:
    server.terminate()
    server.wait(timeout=30)
if 'server_log' in globals():
    server_log.close()
if 'sentence_ui' in globals():
    sentence_ui.close()
print('Backend stopped. GPU memory released.')
